# 06. Test the QED prediction

Compare a photon-only e+e− → μ+μ− sample with the tree-level QED prediction.
Use symmetric beam energies, no shower, and the beam convention specified in your
assignment. The angular code assumes the incoming electron travels along +z.

Choose a symmetric angular acceptance fully covered by the generator cuts.
A full Standard Model sample near the Z is not a photon-only prediction.

Copy this notebook into `work`, select **Python (hep)**, and run cells from the
top. Use the sample and scan points specified in your assignment.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from findingz.catalog import load_catalog
from findingz.delphes import default_run_root
from findingz.hypotheses import build_sample_library

library = build_sample_library(load_catalog(), default_run_root())
display(pd.DataFrame([{"sample_id": k, "label": s.label, "cross_section_pb": s.cross_section_pb,
                      "events": s.generated_events, "config": s.config} for k, s in library.items()]))
def get_sample(sample_id):
    if sample_id not in library:
        raise ValueError(f"Sample {sample_id!r} is unavailable. Choose an ID from the table above.")
    return library[sample_id]
def weights(frame):
    return pd.to_numeric(frame.get("weight", pd.Series(1., index=frame.index)))


In [ ]:
def four_vector(frame, prefix):
    pt, eta, phi, mass = [frame[prefix + "_" + x].to_numpy() for x in ("pt", "eta", "phi", "mass")]
    px, py, pz = pt*np.cos(phi), pt*np.sin(phi), pt*np.sinh(eta)
    return np.column_stack([np.sqrt(px*px+py*py+pz*pz+mass*mass), px, py, pz])
def pair_observables(frame):
    a, b = four_vector(frame, "l1"), four_vector(frame, "l2")
    total = a+b
    mass = np.sqrt(np.maximum(total[:,0]**2 - (total[:,1:]**2).sum(axis=1), 0))
    negative = np.where((frame.l1_charge.to_numpy() < 0)[:,None], a, b)
    momentum = np.linalg.norm(negative[:,1:], axis=1)
    cosine = np.divide(negative[:,3], momentum, out=np.full_like(momentum,np.nan), where=momentum>0)
    return mass, np.hypot(total[:,1],total[:,2]), cosine, total


In [ ]:
sample_id = None
cos_acceptance = 0.8
alpha = 1/137.035999084  # replace with the coupling convention in the actual parameter card
# Fill from the scan run manifests for this assignment. Generator integration errors are NOT experimental errors.
scan = []  # [{"sqrt_s_gev":40., "sigma_pb":..., "integration_error_pb":...}, ...]
if sample_id:
    frame = get_sample(sample_id).load()
    _, _, cosine, _ = pair_observables(frame)
    selected = np.isfinite(cosine) & (abs(cosine)<cos_acceptance)
    bins = np.linspace(-cos_acceptance,cos_acceptance,17)
    hist, _ = np.histogram(cosine[selected],bins=bins,weights=weights(frame)[selected])
    c = cos_acceptance
    primitive = lambda x: x+x**3/3
    probabilities = np.diff(primitive(bins))/(2*c+2*c**3/3)
    centers=(bins[:-1]+bins[1:])/2
    plt.step(centers,hist/hist.sum(),where="mid",label="simulation")
    plt.plot(centers,probabilities,"o",label="QED integrated per bin")
    plt.xlabel("cos θ(mu−)"); plt.ylabel("Fraction within acceptance"); plt.legend(); plt.show()
if scan:
    table=pd.DataFrame(scan)
    s=table.sqrt_s_gev**2
    table["inclusive_massless_qed_pb"]=4*np.pi*alpha**2/(3*s)*3.89379338e8
    display(table)
    print("Apply the same angular/generator acceptance before comparing absolute rates.")

## Questions and submission
1. Derive the normalized angular prediction and its integral over each histogram bin.
2. Determine whether angular acceptance or a lepton-pT cut changes the comparison.
3. Test 1/s scaling; use the actual MadGraph electromagnetic coupling for absolute normalization.
4. Identify the approximations: massless limit, tree level, photon exchange, no radiation.

Submit your edited notebook with figures, units, sample IDs, generation settings, and a short interpretation. Do not equate Monte Carlo event count with experimental luminosity.